上一篇，咱们把 Claude Code 的心脏拆了出来：一个 while 循环、一个 bash 工具、一本只追加的账本，142 行 Python，模型就能在你的终端里自己敲命令、自己看报错、自己修到通。

但那台内核有个一眼可见的糙：模型手里只有一把 bash。想读文件，得拼 cat；想写文件，得拼 echo；想改文件，得拼 sed。模型脑子里想的是"把这段代码写进文件"，出口却只有一条 shell 命令可走。

这一篇就干一件事：把 1 个工具扩成 5 个——bash、read_file、write_file、edit_file、glob。过程中你会亲眼看到一个有点反直觉的事实：工具翻了五倍，主循环一行都不用改。

上一篇结尾我留了四个问题：
1.echo 写文件到底怎么翻车的？
2.模型会不会一轮同时调好几个工具？
3.同时调的工具会不会互相踩？
4.工具的 description 到底要怎么写模型才"懂事"？
这篇全部还掉。

读完你会拿到三样东西：

1.查表分发这套架构：往后加任何新工具，都是"加两行"的事，循环永不再动

2.四个文件工具的完整实现，外加一道把路径锁死在工作区里的安全围栏

3.多工具调用的真实行为观察，和一套写工具描述的实用心法

门槛不变：会 Python 基础语法、有上篇跑通的那份代码。直接开始。

PART 01：还债——用 echo 写文件，到底有多容易翻车
先还第一笔债：给 bash 当唯一工具，写文件到底有多痛。

看一个真实场景。你对上一版的 Agent 说：

帮我创建一个 config.py,内容是:MSG = "It's a $test"
模型给出的命令大概率长这样：

echo 'MSG = "It'"'"'s a $test"' > config.py
没看懂这串鬼画符？正常，我也看不懂。它想用单引号包裹整体，但内容里恰好有个单引号，于是 shell 语法要求把字符串切碎、中间单独转义、再拼回去。这是一个会呼吸的 bug：引号少一层，文件内容就坏一段。

就算模型侥幸拼对了，$test 这一关还在后面等着——双引号里 $test 会被 shell 当变量展开，写进文件的实际是 MSG = "It's a "。$ 没了，变量名没了，你跑代码之前根本发现不了。

最阴险的就在这：bash 写文件，坏了是不报错的。 命令执行成功、退出码为 0、工具返回"(no output)"，模型满心以为写好了，直到哪天程序跑炸，你翻出文件一看，内容早被 shell 搞得面目全非。报错不可怕，报错是反馈；静默出错才可怕，它把坑埋进未来。

多行内容更是一场灾难。echo 天生只吃单行，想写个十行的脚本，模型得拼十条命令、或者玩 -e 加 \n 转义，每一步都在给出错概率充值。

把这些坑摆在一起，你会发现它们其实是同一个病：

模型想的是"写这些内容"，却被逼先当一遍 shell 语法翻译官。 意图和动作之间，隔了一整层命令行转义。每次翻译，都是一次出错机会；每次防错，都是一圈多余 token。

读文件同理：模型想要"前 50 行"，cat 给不了，得 head -n 50；想知道"改哪一行"，sed 的正则方言分分钟能让模型当场翻车。

解法已经写在问题里了：它想读，就直接给它 read；想写，就直接给它 write；想找文件，就直接给它 glob。 让意图直达动作，把翻译层整个拆掉。

怎么拆，而不把上篇写好的循环拆坏？看下一部分，一场只动一行的手术。